# CranioVision — Attention U-Net Training

**Branch:** `feature/attention-unet`  
**Model:** 3D Attention U-Net (MONAI)  
**Dataset:** BraTS 2024 small (140 cases)  
**Runtime target:** Kaggle T4 GPU (~2-4 hours for 100 epochs)

---

## What this notebook does

1. Imports shared code from `src/cranovision/` (config, data, models, training)
2. Loads the BraTS dataset with train/val/test splits
3. Builds MONAI `CacheDataset` loaders with 4-modality stacking
4. Instantiates a 3D Attention U-Net
5. Trains with `DiceCELoss`, `AdamW`, `CosineAnnealingLR`
6. Validates every 5 epochs with sliding-window inference
7. Saves best checkpoint to `models/attention_unet_best.pth`
8. Plots training curves and visualizes predictions

**Where to run:** This notebook is designed to run on Kaggle (T4 GPU) via a launcher notebook. It can also run locally for testing if your GPU is big enough.

## 1. Environment Setup

Handles path setup for both local and Kaggle environments automatically.

In [1]:
import sys
import os
from pathlib import Path

# ── Detect environment and set project root ──
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    # Running on Kaggle — repo was cloned to /kaggle/working/CranioVision
    PROJECT_ROOT = Path('/kaggle/working/CranioVision')
    IS_KAGGLE = True
else:
    # Running locally — notebook is in CranioVision/notebooks/
    PROJECT_ROOT = Path.cwd().parent
    IS_KAGGLE = False

# Add project root to Python path so 'from src.cranovision import ...' works
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Environment : {"KAGGLE" if IS_KAGGLE else "LOCAL"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Exists      : {PROJECT_ROOT.exists()}')

Environment : LOCAL
Project root: d:\2_ML PROJECTS\30. Brainstorm\CranioVision
Exists      : True


In [2]:
# ── For Kaggle only: override data path to attached dataset ──
if IS_KAGGLE:
    # Replace with the actual path where you attached your Kaggle dataset
    # Attach it in the Kaggle notebook sidebar → 'Add Input'
    # The path is typically: /kaggle/input/<dataset-slug>/<folder-inside>/
    import src.cranovision.config as cfg
    
    # Auto-detect the dataset folder under /kaggle/input
    kaggle_input = Path('/kaggle/input')
    candidates = list(kaggle_input.glob('*/BraTS2024_small_dataset'))
    if not candidates:
        # Fallback: look for any folder containing BraTS- prefixed subfolders
        for d in kaggle_input.iterdir():
            if d.is_dir():
                subs = list(d.glob('BraTS-*'))
                if subs:
                    candidates = [d]
                    break
    
    if candidates:
        cfg.RAW_DATA_DIR = candidates[0]
        print(f'✅ Kaggle dataset detected: {cfg.RAW_DATA_DIR}')
    else:
        raise FileNotFoundError(
            'Could not locate BraTS dataset under /kaggle/input. '
            'Attach your dataset via the Kaggle notebook sidebar → Add Input.'
        )
    
    # Also update checkpoint/output dirs to /kaggle/working (writable)
    cfg.MODELS_DIR  = Path('/kaggle/working/models')
    cfg.OUTPUTS_DIR = Path('/kaggle/working/outputs')
    cfg.SPLITS_DIR  = Path('/kaggle/working/splits')
    for d in (cfg.MODELS_DIR, cfg.OUTPUTS_DIR, cfg.SPLITS_DIR):
        d.mkdir(parents=True, exist_ok=True)
    print(f'Models dir  : {cfg.MODELS_DIR}')
    print(f'Outputs dir : {cfg.OUTPUTS_DIR}')

In [3]:
# Now import CranioVision modules
from src.cranovision.config import (
    RAW_DATA_DIR, MODELS_DIR, OUTPUTS_DIR, DEVICE, USE_AMP,
    PATCH_SIZE, BATCH_SIZE, NUM_WORKERS, PIN_MEMORY,
    LEARNING_RATE, WEIGHT_DECAY, MAX_EPOCHS, VAL_INTERVAL,
    NUM_CLASSES, print_config,
)
from src.cranovision.data import (
    get_splits, get_train_transforms, get_val_transforms,
)
from src.cranovision.models import build_attention_unet, count_parameters
from src.cranovision.training import TrainConfig, train

print_config()

c:\Users\hrana\anaconda3\envs\ml_env_fixed\Lib\site-packages\ignite\handlers\checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


════════════════════════════════════════════════════════════
CranioVision Configuration
════════════════════════════════════════════════════════════
Project root : D:\2_ML PROJECTS\30. Brainstorm\CranioVision
Data dir     : D:\2_ML PROJECTS\30. Brainstorm\CranioVision\data\raw\BraTS2024_small_dataset
Models dir   : D:\2_ML PROJECTS\30. Brainstorm\CranioVision\models
Outputs dir  : D:\2_ML PROJECTS\30. Brainstorm\CranioVision\outputs
Device       : cuda
Mixed precision: True
Patch size   : (128, 128, 128)
Num classes  : 4
Modalities   : ['t1n', 't1c', 't2w', 't2f']
════════════════════════════════════════════════════════════


## 2. Data Loading

Uses the shared `get_splits()` from `src.cranovision.data.dataset`.  
The split is saved to `data/splits/data_split.json` — **same split is used by all 3 model branches** for fair comparison.

In [ ]:
train_files, val_files, test_files = get_splits()

In [ ]:
from monai.data import CacheDataset, DataLoader

train_transforms = get_train_transforms()
val_transforms   = get_val_transforms()

print('Building CacheDatasets (this caches all data in RAM for fast epochs)...')
print('First run preprocesses all cases — takes ~5-10 min. After that, epochs are fast.\n')

train_ds = CacheDataset(data=train_files, transform=train_transforms,
                         cache_rate=1.0, num_workers=NUM_WORKERS)
val_ds   = CacheDataset(data=val_files,   transform=val_transforms,
                         cache_rate=1.0, num_workers=NUM_WORKERS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                           num_workers=0, pin_memory=PIN_MEMORY)

print(f'\n✅ Loaders ready.')
print(f'  Train batches: {len(train_loader)}')
print(f'  Val batches  : {len(val_loader)}')

## 3. Model — 3D Attention U-Net

In [ ]:
model = build_attention_unet().to(DEVICE)

info = count_parameters(model)
print(f'Attention U-Net built on {DEVICE}')
print(f'  Total params    : {info["total"]:>12,}  ({info["total_M"]:.2f}M)')
print(f'  Trainable params: {info["trainable"]:>12,}')

# Shape sanity check
import torch
with torch.no_grad():
    dummy = torch.randn(1, 4, *PATCH_SIZE).to(DEVICE)
    out = model(dummy)
print(f'\nShape check:')
print(f'  Input  : {tuple(dummy.shape)}')
print(f'  Output : {tuple(out.shape)}  (should be (1, 4, {"×".join(str(s) for s in PATCH_SIZE)}))')
del dummy, out
torch.cuda.empty_cache()

## 4. Loss, Optimizer, Scheduler

- **Loss:** `DiceCELoss` = Dice + CrossEntropy (handles class imbalance)
- **Optimizer:** `AdamW` (proper weight decay)
- **Scheduler:** `CosineAnnealingLR` (smooth LR decay)

In [ ]:
from monai.losses import DiceCELoss

loss_fn = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    lambda_dice=1.0,
    lambda_ce=1.0,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
    eta_min=1e-6,
)

print(f'Loss     : DiceCELoss')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingLR (T_max={MAX_EPOCHS})')

## 5. Train

Uses the shared `train()` function from `src.cranovision.training.trainer`.
Best checkpoint is saved automatically as `models/attention_unet_best.pth`.

In [ ]:
train_config = TrainConfig(
    model_name='attention_unet',
    max_epochs=MAX_EPOCHS,
    val_interval=VAL_INTERVAL,
    patch_size=PATCH_SIZE,
    use_amp=USE_AMP,
    ckpt_dir=MODELS_DIR,
    history_dir=OUTPUTS_DIR,
)

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    config=train_config,
)

## 6. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for s in ax.spines.values():
        s.set_edgecolor('#444')

# Loss curve
axes[0].plot(range(1, len(history.train_loss) + 1), history.train_loss,
              color='#FF6B35', linewidth=2, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('DiceCE Loss')
axes[0].set_title('Training Loss')
axes[0].legend(facecolor='#2a2a2a', labelcolor='white')
axes[0].grid(alpha=0.2)

# Validation Dice
axes[1].plot(history.val_epochs, history.val_dice,
              color='#4ECDC4', linewidth=2, marker='o', markersize=5,
              label='Mean Val Dice')
axes[1].axhline(y=history.best_dice, color='#FFD700', linestyle='--',
                 alpha=0.7, label=f'Best: {history.best_dice:.4f} (ep {history.best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Dice (excl. background)')
axes[1].set_title('Validation Dice')
axes[1].legend(facecolor='#2a2a2a', labelcolor='white')
axes[1].grid(alpha=0.2)
axes[1].set_ylim(0, 1)

plt.suptitle('CranioVision — Attention U-Net Training',
              color='white', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'attention_unet_curves.png', dpi=150,
             bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f'Saved: {OUTPUTS_DIR / "attention_unet_curves.png"}')

## 7. Summary

In [ ]:
print('=' * 60)
print('ATTENTION U-NET — TRAINING COMPLETE')
print('=' * 60)
print(f'Best epoch     : {history.best_epoch}')
print(f'Best Val Dice  : {history.best_dice:.4f}')
print(f'Total epochs   : {MAX_EPOCHS}')
print(f'Checkpoint     : {MODELS_DIR / "attention_unet_best.pth"}')
print(f'History JSON   : {OUTPUTS_DIR / "attention_unet_history.json"}')
print(f'Curves PNG     : {OUTPUTS_DIR / "attention_unet_curves.png"}')
print('=' * 60)
print('\nNext step: download the .pth file + history JSON from Kaggle Output,')
print('place in local models/ folder, then proceed with inference notebooks.')